In [1]:
# ===== Install dependencies =====
import pandas as pd
import numpy as np
import re

In [2]:
# ===== Missing values processing =====
data = pd.read_csv('data\\profiles.csv', index_col = False)
print(data.isna().sum())

Name                     0
Age                      0
Status                   0
Experience               0
Rating                4171
N_reviews             4171
N_last_month_views       0
Last_activity_date       0
N_students_placed        0
Description           2197
Price                    0
dtype: int64


In [3]:
data.Rating = data.Rating.fillna('рейтинг отсутствует')
data.N_reviews = data.N_reviews.fillna('нет отзывов')
data.Description = data.Description.fillna('Дополнительные сведения отсутствуют')

In [4]:
# ===== Price processing =====
# Here we will normalize a price
# to scale 'rubles per one hour'
data[~data.Price.str.contains('₽')].Price.value_counts()

Price
―                  164
15 Br/60 минут       3
30 Br/60 минут       3
10 Br/60 минут       2
102 Br/60 минут      2
41 Br/60 минут       2
54 Br/60 минут       1
34 Br/60 минут       1
40 Br/60 минут       1
46 Br/45 минут       1
21 Br/60 минут       1
25 Br/60 минут       1
24 Br/45 минут       1
30 Br/45 минут       1
20 Br/60 минут       1
43 Br/60 минут       1
42 Br/45 минут       1
20 Br/40 минут       1
44 Br/60 минут       1
51 Br/60 минут       1
78 Br/60 минут       1
31 Br/60 минут       1
80 Br/60 минут       1
Name: count, dtype: int64

In our sample there are some tutors who do not offer online lessons and few ones from Belarus. We won't consider these cases in futher analysis.

In [5]:
data_cleaned = data[data.Price.str.contains('₽')]

normalized_price_list = []
for price in data_cleaned.Price:
    money = int(re.search(r'\b([\d\s]+)₽', price).group(1).replace(' ', ''))
    duration = int(re.search(r'₽/(\d+)\sминут', price).group(1))
    price_normalized = round(money / duration * 60)
    normalized_price_list.append(price_normalized)

data_cleaned['Normalized_price'] = normalized_price_list
data_cleaned = data_cleaned.drop('Price', axis = 1)

In [ ]:
# ===== Combine information into a single description =====
def form_additional_information(row: pd.Series) -> str:

    name = row.Name
    age = row.Age
    status = row.Status
    experience = row.Experience
    rating = row.Rating
    reviews = row.N_reviews
    views = row.N_last_month_views
    activity = row.Last_activity_date
    placed = row.N_students_placed

    additional_information = f'''
    Имя: {name}
    Возраст: {age}
    Статус: {status}
    Стаж преподавания: {experience}
    Рейтинг: {rating}
    Количество отзывов: {reviews}
    Просмотры анкеты за последний месяц: {views}
    Последняя активность: {activity}
    Количество подобранных студентов: {placed}
    '''

    return additional_information

    
unified_descriptions = []
n_rows = data_cleaned.shape[0]

for i in range(n_rows):
    new_row = data_cleaned.iloc[i, ]
    add_info = form_additional_information(new_row)
    uni_desc = new_row.Description + add_info
    unified_descriptions.append(uni_desc)

data_cleaned['Unified_description'] = unified_descriptions
processed_data = data_cleaned[['Unified_description', 'Normalized_price']]
processed_data.to_csv('processed_data.csv', index = False)

In [7]:
# ===== Encoding of target value =====
normalized_price = data_cleaned.Normalized_price
normalized_price.describe()
quantiles = normalized_price.quantile([0.2, 0.4, 0.6, 0.8, 0.9])
quantiles

0.2    1000.0
0.4    1200.0
0.6    1500.0
0.8    2000.0
0.9    2500.0
Name: Normalized_price, dtype: float64

In [14]:
bins = [0, 1000, 1500, 2000, 2500, np.inf]
labels = [1, 2, 3, 4, 5]
price_categories = pd.cut(normalized_price, bins=bins, labels=labels)
encoded_price = pd.get_dummies(price_categories)
np.save('target.npy', np.array(encoded_price.astype(int)))